# Spark Connector

In [14]:
!pip cache purge --quiet

In [15]:
!pip install install-jdk==1.1.0 \
             pyspark==4.0.0 --quiet

In [16]:
import jdk
import os

from pyspark.sql import SparkSession
from singlestoredb.management import get_secret

os.environ.pop("CONTAINER_ID", None)

'3bd646e5-a98e-4f84-a6ca-fea7fc21dad3'

In [17]:
# Configure Java
jdk_path = jdk.install("17")

os.environ["JAVA_HOME"] = jdk_path
os.environ["PATH"] = f"{jdk_path}/bin:" + os.environ["PATH"]

In [18]:
# Define fully-qualified Maven coordinates for JARs
jar_packages = [
    "com.singlestore:singlestore-spark-connector_2.13:4.2.0-spark-4.0.0",
    "com.singlestore:singlestore-jdbc-client:1.2.5",
    "org.apache.commons:commons-dbcp2:2.12.0",
    "org.apache.commons:commons-pool2:2.12.0",
    "io.spray:spray-json_2.13:1.3.6"
]

spark = (
    SparkSession.builder
    .appName("Spark Connector")
    .master("local[*]")
    .config("spark.jars.packages", ",".join(jar_packages))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
com.singlestore#singlestore-spark-connector_2.13 added as a dependency
com.singlestore#singlestore-jdbc-client added as a dependency
org.apache.commons#commons-dbcp2 added as a dependency
org.apache.commons#commons-pool2 added as a dependency
io.spray#spray-json_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-927848c1-c69b-4969-8931-c991b7245ab6;1.0
	confs: [default]
	found com.singlestore#singlestore-spark-connector_2.13;4.2.0-spark-4.0.0 in central
	found org.apache.avro#avro;1.11.3 in central
	found com.fasterxml.jackson.core#jackson-core;2.14.2 in central
	found com.fasterxml.jackson.core#jackson-databind;2.14.2 in central
	found com.fasterxml.jackson.core#jackson-annotat

In [19]:
# Create a DataFrame
data = [("Peter", 27), ("Paul", 28), ("Mary", 29)]
df = spark.createDataFrame(data, ["Name", "Age"])

In [20]:
# Show the content of the DataFrame
df.show()

+-----+---+
| Name|Age|
+-----+---+
|Peter| 27|
| Paul| 28|
| Mary| 29|
+-----+---+



<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the workspace from the drop-down menu at the top of this notebook.</p>
    </div>
</div>

In [21]:
%%sql
%%sql
CREATE DATABASE IF NOT EXISTS spark_demo_db;

1 rows affected.

++
||
++
++

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [24]:
from sqlalchemy import *

db_connection = create_engine(connection_url)
url = db_connection.url

In [25]:
password = get_secret("password")
database = url.database
host = url.host
port = url.port
cluster = host + ":" + str(port)

In [26]:
spark.conf.set("spark.datasource.singlestore.ddlEndpoint", cluster)
spark.conf.set("spark.datasource.singlestore.user", "admin")
spark.conf.set("spark.datasource.singlestore.password", password)
spark.conf.set("spark.datasource.singlestore.disablePushdown", "false")

In [27]:
(df.write
    .format("singlestore")
    .option("loadDataCompression", "LZ4")
    .mode("overwrite")
    .save(f"{database}.demo")
)

In [28]:
new_df = (spark.read
    .format("singlestore")
    .load(f"{database}.demo")
)

In [29]:
# Show the content of the DataFrame
new_df.show()

+-----+---+
| Name|Age|
+-----+---+
| Paul| 28|
| Mary| 29|
|Peter| 27|
+-----+---+



In [30]:
# Stop the Spark session
spark.stop()